# 00 · Colab setup and environment check

Mounts Drive, clones the repo and installs the pinned requirements. Then it checks the GPU and logs the exact library versions to Drive. **Runtime → Change runtime type → GPU** first.

In [1]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar).

    The token goes in a one-off HTTP header, so it is never written to .git/config."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth = []
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = ["-c", f"http.https://github.com/.extraheader=Authorization: Basic {basic}"]
    res = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    out = res.stdout + res.stderr
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH))
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

Mounted at /content/drive
repo: /content/ie7615-generative-project | commit: a8e10f6


In [8]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()

# Safely ensure expected Milestone 2 and model keys are in the config path list to prevent KeyError
p = cfg.setdefault("paths", {})
p.setdefault("milestone2_samples", "outputs/milestone2_samples")
p.setdefault("milestone2_logs", "outputs/milestone2_logs")
p.setdefault("bert_adapter_ckpt", "checkpoints/bert_to_clip_adapter.pt")

paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

artifact root: /content/drive/MyDrive/ie7615-generative


In [5]:
import torch
from src.utils import environment_info, save_json

env = environment_info()
for k, v in env.items():
    print(f"{k}: {v}")
if not torch.cuda.is_available():
    print("\nNo GPU found. Switch the runtime to GPU before running notebooks 02 and 03.")
else:
    free, total = torch.cuda.mem_get_info()
    print(f"\nGPU memory: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")

python: 3.13.16
packages: {'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'diffusers': '0.42.1', 'transformers': '5.22.1', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'pycocotools': '2.0.11', 'numpy': '2.4.6', 'pillow': '11.3.0'}
cuda_available: True
gpu: Tesla T4
cuda: 13.0

GPU memory: 15.5 GB free of 15.6 GB


In [6]:
# Record the environment of this session (versions, GPU) next to the results.
print("saved:", save_json({"notebook": "00_colab_setup"}, paths["outputs"] / "env_00_colab_setup.json", add_env=True))

saved: /content/drive/MyDrive/ie7615-generative/outputs/env_00_colab_setup.json


In [7]:
# Optional: pre-download the model weights into the HF cache (~3 GB total) to speed up 02 and 03.
from huggingface_hub import snapshot_download
m = cfg["models"]
for key in ("clip", "bert"):
    snapshot_download(m[key], revision=m["revisions"][key], allow_patterns=["*.json", "*.txt", "*.safetensors"])
snapshot_download(m["diffusion"], revision=m["revisions"]["diffusion"],
                  allow_patterns=["*.json", "*.txt", "*fp16.safetensors"])
print("model weights cached")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

model weights cached
